In [1]:
import json, random
from pathlib import Path

input_path = '../../data/3-task/totto.json'
output_path = '../../data/4-rlvr/totto.json'

# Change this if you want reproducibility (e.g., seed=42). None = true random each run.
seed = None
train_ratio = 0.8

with open(input_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Locate the list of samples
samples = None
container = None  # reference to the object that contains the list
key_used = None

if isinstance(data, list):
    samples = data
    container = None  # top-level list
elif isinstance(data, dict):
    for k in ['data', 'examples', 'items', 'rows', 'samples']:
        v = data.get(k)
        if isinstance(v, list):
            samples = v
            container = data
            key_used = k
            break

if samples is None:
    raise ValueError("Could not find a list of samples. Expected a top-level list or a dict with a list under one of: data/examples/items/rows/samples.")

n = len(samples)
idxs = list(range(n))
rng = random.Random(seed)  # Random() allows optional seed for reproducibility
rng.shuffle(idxs)

cut = int(train_ratio * n)  # floor
train_set = set(idxs[:cut])

# Add/overwrite the 'split' key
for i, ex in enumerate(samples):
    ex['split'] = 'train' if i in train_set else 'test'

# Write output (preserve original structure)
Path(output_path).parent.mkdir(parents=True, exist_ok=True)
with open(output_path, 'w', encoding='utf-8') as f:
    json.dump(data, f, ensure_ascii=False, indent=2)

print(f"Total: {n} | train: {len(train_set)} | test: {n - len(train_set)}"
      + ("" if key_used is None else f" | list key: '{key_used}'"))


Total: 1000 | train: 800 | test: 200
